# CFHT AEONLib Demonstration Notebook
The online tests are also a good reference and are more extensive. See tests/cfht/test_online.py

In [17]:
import random
# Import generic Aeonlib models, as well as CFHT facilties
from aeonlib.cfht.facility import CFHTFacility
from aeonlib.cfht.conversions import target_data_from_aeon
from aeonlib.cfht.models import Instrument, TargetDataMagnitude
from aeonlib.conf import settings
from aeonlib.models import SiderealTarget

## Initialize the CFHT facility and select an observing program

In [18]:
# Normally credentials are automatically picked up by the environment. For this notebook we supply them directly
settings.cfht_access_token = "<your api token here>"
settings.cfht_api_root = "https://api-stage.cfht.hawaii.edu/"
facility = CFHTFacility(settings=settings)

# Get a list of Programs available to us, and select the one we wish to use. Alternatively, the facility can be
# instatiated with a program token directly, which is probably what you'd want to use in normal circumstances.
programs = facility.programs()
if not programs:
    raise ValueError("No programs found. Check the Kealahou Phase2 Tool")
# Set the first available program as the active one
program = programs[0]
facility.select_program(program)

## Define a target
We use a generic AEONLib target as a starting point here, as it makes it easier to share between other non-CFHT facilities. If CFHT is the only facility you will be using, it might be easier to construct an aeonlib.cfht.models.TargetData instance directly.

In [19]:
# Start with the AEONLib target
sidereal_target = SiderealTarget(
    name="AEONLib CFHT Notebook Target",
    type="ICRS",
    ra=320.11,
    dec=-42.0
)
# Use it to bootstrap a full CFHT TargetData
target_data = target_data_from_aeon(sidereal_target)
# Fill in CFHT specific data
target_data.token = f"{facility.program_token}-{random.randint(1000000000, 9999999999)}"
target_data.magnitude = TargetDataMagnitude(ab=10.0)
target_data.temperature_effective = 1234.5
target_data.standard_star = False
target_data.pointing_offset_token = f"00AZ00-PO+{Instrument.megacam.value}+1"

## Create and list targets

In [20]:
target = facility.create_or_update_target(target_data, Instrument.megacam)
all_targets = facility.targets()
try:
    found_target = next(t for t in all_targets if t.token == target.token)
    print(found_target)
except StopIteration:
    raise ValueError("Created target did not appear in target list!")

token='25BE25-5950669302' name='AEONLib CFHT Notebook Target' label=7 version=1 fixed_target=TargetDataFixedTarget(coordinate=SkyCoordinate(ra=320.11, dec=-42.0), proper_motion=FixedTargetProperMotion(ra_mas=None, dec_mas=None), computed_coordinate=None, estimated_radial_velocity_kmps=None) moving_target=None magnitude=TargetDataMagnitude(u=None, b=None, v=None, r=None, i=None, g=None, j=None, h=None, k=None, uu=None, gg=None, rr=None, ii=None, zz=None, ab=10.0) temperature_effective=1234.5 standard_star=None linked_target_identifiers=None finding_chart=[] pointing_offset_token='00AZ00-PO+MEGACAM+1' pointing_offset=PointingOffsetData(token='00AZ00-PO+MEGACAM+1', name='1', offset=OffsetCoordinate(ra_offset=None, dec_offset=None, exposure_number=None), label=None, version=None, user_token='SYSTEM', instrument=<Instrument.megacam: 'MEGACAM'>, is_system=True)


## Delete created target
Now would be a good time to check the K2 tool in your browser to confirm the target appears in the targets section before it is deleted.

In [21]:
if target.token is not None:
    facility.delete_target(target.token)